In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
#  OWT CLASSIFICATION  (Uudeberg et al. 2019, §2.3-2.4)
#  Stage 1: rule-based first pass on spectral shape (max location, amplitude, slope)
#  Stage 2: average rule-classified spectra -> full-resolution reference curves
#  Stage 3: final label = argmax of delta_j = 10*SCS_j + (1-MSAS_j)/2   (Eq. 3)
# ─────────────────────────────────────────────────────────────────────────────

def classify_owt_rules(rrs, wl_full):
    """First-pass label from spectral-shape rules (paper §2.3, §3.1)."""
    mask = (wl_full >= 400) & (wl_full <= 850)
    wl_sub, r_sub = wl_full[mask], rrs[mask]
    if np.all(np.isnan(r_sub)):
        return 'Unknown'

    interp = interp1d(wl_full, rrs, kind='linear', bounds_error=False, fill_value=np.nan)
    wl_max = wl_sub[np.nanargmax(r_sub)]
    r_max  = np.nanmax(r_sub)
    r500   = float(interp(500))
    r650   = float(interp(650))

    if r_max < 0.006 and wl_max > 650:
        return 'Brown'
    if 685 <= wl_max <= 715:
        return 'Very_Turbid'
    if 580 <= wl_max < 605:
        return 'Turbid'
    if 540 <= wl_max < 580:
        if not np.isnan(r500) and not np.isnan(r650) and r500 > r650:
            return 'Clear'
        return 'Moderate'
    return 'Unknown'


def build_owt_references(spec_df, wl_cols, wavelengths, min_n=5):
    """Average rule-classified spectra into full-resolution OWT reference curves."""
    labels = [classify_owt_rules(row[wl_cols].values.astype(float), wavelengths)
              for _, row in spec_df.iterrows()]
    tmp = spec_df.copy()
    tmp['_owt_rule'] = labels

    refs, counts = {}, {}
    for owt in OWT_NAMES:
        sub = tmp[tmp['_owt_rule'] == owt]
        counts[owt] = len(sub)
        refs[owt] = sub[wl_cols].values.astype(float).mean(axis=0) if len(sub) >= min_n else None

    print('  Rule-based first-pass counts:', counts)
    missing = [k for k, v in refs.items() if v is None]
    if missing:
        print(f'  WARNING: no reference built for {missing} '
              f'(fewer than {min_n} rule-classified spectra) — these OWTs are '
              f'unreachable in the final classification until more data is added.')
    return refs, counts


def classify_owt_similarity(rrs, wl_full, refs, wl_lo=400, wl_hi=900):
    """delta_j = 10*SCS_j + (1 - MSAS_j)/2   (Uudeberg et al. 2019, Eq. 3)."""
    mask = (wl_full >= wl_lo) & (wl_full <= wl_hi)
    vec = rrs[mask]
    if np.all(np.isnan(vec)) or np.nansum(np.abs(vec)) == 0:
        return 'Unknown', {}

    scores = {}
    for name, ref_full in refs.items():
        if ref_full is None:
            continue
        ref_vec = ref_full[mask]
        valid = ~(np.isnan(vec) | np.isnan(ref_vec))
        if valid.sum() < 5:
            continue
        v, r = vec[valid], ref_vec[valid]
        try:
            scs, _ = pearsonr(v, r)
        except Exception:
            scs = -1.0
        denom = np.linalg.norm(v) * np.linalg.norm(r) + 1e-12
        cos  = np.clip(np.dot(v, r) / denom, -1.0, 1.0)
        msas = 1.0 - np.arccos(cos) / np.pi
        scores[name] = 10.0 * scs + (1.0 - msas) / 2.0

    if not scores:
        return 'Unknown', {}
    best = max(scores, key=scores.get)
    return best, scores
